In [2]:
import polars as pl
from huggingface_hub import hf_hub_download 

local_path = hf_hub_download(
    repo_id="acon96/Home-Assistant-Requests-V2",
    filename="home_assistant_test.jsonl", # NOTE: we download test set because train set is too big to download
    repo_type="dataset"
)

df = pl.read_ndjson(local_path)
df.row(0)[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': "You are 'Al', a helpful AI Assistant that controls the devices in a house. Complete the following task as instructed or answer the following question with the information provided only.\nThe current time and date is 8:12 AM on Saturday, April 17 2027\nDevices:\nswitch.water_feature_lights 'Water Feature Lights Switch' = off\nlight.front_courtyard_homekit 'Front Courtyard Light' = on;mediumorchid (191, 63, 221);80%\nlight.front_balcony_homekit 'Front Balcony Light' = off;limegreen (46, 211, 85);92%\nlight.downstairs_master_bedroom_homekit 'Downstairs Master Bedroom Light' = on;mediumspringgreen (12, 234, 173);65%\ntimer.laundry_countdown 'Laundry cycle timer' = active\nlight.backyard 'Backyard Light' = on;74%\nlight.front_stairwell_ge 'Front Stairwell Light' = on;87%\nmedia_player.apple_tv_4k 'Apple 4K Streaming Device' = off\nfan.craft_room 'Craft room fan' = on\nlock.mudroom_entry 'Mudroom entry lock' = unlocked\nfan.bedr

In [3]:
from tinyreasoner import chat_api
import json
def _process_message(message: dict):
    if message["role"] == "tool":
        tc = json.loads(message["content"][0]["text"])
        return {"role": "tool", "name": tc["tool_name"], "content": tc["tool_result"]}
    return message

def load(verbose:bool=True):
    local_path = hf_hub_download(
        repo_id="acon96/Home-Assistant-Requests-V2",
        filename="home_assistant_test.jsonl", # NOTE: we download test set because train set is too big to download
        repo_type="dataset"
    )

    df = pl.read_ndjson(local_path)
    chats = []
    for messages, tools in df.iter_rows():
        messages = [_process_message(m) for m in messages]
        chats.append({"tools": tools, "messages": messages})

    samples = chat_api.convert_dataset(chats, require_tc_id=False, verbose=verbose)
    return samples

samples = load()
samples[0]

('{\'function\': {\'name\': \'HassMediaPrevious\', \'arguments\': \'{"name": "media_player.kitchen"}\'}} is not defined in {\'HassMediaNext\', \'HassSetVolume\', \'HassCancelTimer\', \'HassUnpauseTimer\', \'HassToggle\', \'HassMediaUnpause\', \'HassHumidifierMode\', \'HassListAddItem\', \'HassSetPosition\', \'HassPauseTimer\', \'HassStartTimer\', \'HassTurnOff\', \'HassTurnOn\', \'HassVacuumStart\', \'HassHumidifierSetpoint\', \'HassVacuumReturnToBase\', \'HassClimateSetTemperature\', \'HassLightSet\', \'HassMediaPause\'}:\n', '{\n    "tools": [\n        {\n            "function": {\n                "name": "HassTurnOn",\n                "description": "Turns on/opens/unlocks a device or entity",\n                "parameters": {\n                    "type": "object",\n                    "properties": {\n                        "name": {\n                            "type": "string",\n                            "description": "Name of the device or entity"\n                        },\

[ToolDefinition(content={"name": "HassTurnOn", "description": "Turns on/opens/unlocks a device or entity", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Name of the device or entity"}, "area": {"type": "string", "description": "Name of the area"}, "floor": {"type": "string", "description": "Name of the floor"}, "color": null, "temperature": null, "brightness": null, "duration": null, "item": null, "humidity": null, "position": null, "volume_level": null, "mode": null, "domain": {"type": "array", "items": {"type": "string"}, "description": "Device domain(s)"}, "device_class": {"type": "array", "items": {"type": "string"}, "description": "Device class(es)"}}, "required": []}}, type=tool_definition),
 ToolDefinition(content={"name": "HassTurnOff", "description": "Turns off/closes/locks a device or entity", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Name of the device or entity"}, "area": {"type": 

In [4]:
len(samples)

15343